# LeetCode 3786 — Total Sum of Interaction Cost in Tree Groups bằng Python

- Link: https://leetcode.com/problems/total-sum-of-interaction-cost-in-tree-groups/
- Độ khó: **Hard**
- Chủ đề: Cây, đếm theo đóng góp của **cạnh**, đếm theo nhóm (≤ 20 nhóm), DP cây con, đổi gốc (rerooting)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def interactionCosts(self, n: int, edges: List[List[int]], group: List[int]) -> int:
```

## 1. Mục tiêu học

1. Đổi "tổng khoảng cách mọi cặp" thành "tổng **đóng góp của từng cạnh**": cạnh nằm trên đường đi của cặp `(u, v)` ⇔ cạnh tách `u` và `v`.
2. Với ràng buộc **cùng nhóm**: cạnh `(v, cha)` đóng góp `Σ_g in_g(v) · (tot_g − in_g(v))`, trong đó `in_g(v)` là số nút nhóm `g` trong cây con của `v`.
3. Tận dụng `group[i] ≤ 20` ⇒ mỗi nút giữ vector đếm độ dài ≤ 20 — `O(20 · n)`.
4. Biết kỹ thuật **đổi gốc** tính tổng khoảng cách từ mọi nút tới các nút của một nhóm.
5. Duyệt cây **không đệ quy** (BFS + thứ tự ngược).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cây `n` nút, nút `i` thuộc nhóm `group[i]`. Chi phí tương tác giữa hai nút là số cạnh trên đường đi giữa chúng. Tính tổng chi phí trên mọi cặp **không thứ tự** `(u, v)`, `u ≠ v`, **cùng nhóm**.

**Ví dụ 1:** đường `0-1-2`, `group = [1,1,1]` → `1 + 1 + 2 = 4`.

**Ví dụ 2:** đường `0-1-2`, `group = [3,2,3]` → `2`.

**Ví dụ 3:** sao tâm 0, `group = [1,1,4,4]` → `1 + 2 = 3`.

**Ví dụ 4:** `n = 2`, `group = [9,8]` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ group[i] ≤ 20`.

In [1]:
from typing import List
from collections import deque, Counter
import random, time

examples = [
    (3, [[0, 1], [1, 2]], [1, 1, 1], 4),
    (3, [[0, 1], [1, 2]], [3, 2, 3], 2),
    (4, [[0, 1], [0, 2], [0, 3]], [1, 1, 4, 4], 3),
    (2, [[0, 1]], [9, 8], 0),
]
for n, e, g, exp in examples:
    print(n, e, g, "-> kỳ vọng", exp)

3 [[0, 1], [1, 2]] [1, 1, 1] -> kỳ vọng 4
3 [[0, 1], [1, 2]] [3, 2, 3] -> kỳ vọng 2
4 [[0, 1], [0, 2], [0, 3]] [1, 1, 4, 4] -> kỳ vọng 3
2 [[0, 1]] [9, 8] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. BFS lấy thứ tự và cha (thay cho DFS đệ quy)

Duyệt BFS từ gốc 0 cho `order` (cha luôn đứng trước con) và `parent`. Duyệt `order` **ngược** thì con được xử lý trước cha — đủ cho DP cây con.

In [2]:
def bfs_order(n, edges, root=0):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    parent = [-1] * n; order = [root]; seen = [False] * n; seen[root] = True
    for u in order:
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; order.append(v)
    return order, parent, adj

print(bfs_order(4, [[0, 1], [0, 2], [2, 3]]))

([0, 1, 2, 3], [-1, 0, 0, 2], [[1, 2], [0], [0, 3], [2]])


### 3.2. Đóng góp của cạnh vào tổng khoảng cách

Tổng khoảng cách mọi cặp = `Σ_cạnh (số cặp mà đường đi qua cạnh)`. Cạnh `(v, cha)` tách cây thành cây con của `v` (`s` nút) và phần còn lại (`n − s` nút) ⇒ đóng góp `s · (n − s)`.

In [3]:
def all_pairs_distance_sum(n, edges):
    order, parent, _ = bfs_order(n, edges)
    size = [1] * n; total = 0
    for u in reversed(order):
        if parent[u] != -1:
            total += size[u] * (n - size[u]); size[parent[u]] += size[u]
    return total

print(all_pairs_distance_sum(3, [[0, 1], [1, 2]]), "(1 + 1 + 2 = 4)")

4 (1 + 1 + 2 = 4)


### 3.3. Nén nhãn nhóm

Nhãn nhóm `1..20` ⇒ dùng mảng đếm độ dài 21, hoặc nén về `0..G−1` với `G` = số nhóm thực sự xuất hiện.

In [4]:
group = [9, 3, 9, 20, 3]
ids = {g: i for i, g in enumerate(sorted(set(group)))}
print(ids, [ids[g] for g in group])

{3: 0, 9: 1, 20: 2} [1, 0, 1, 2, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Áp dụng 3.2 cho từng nhóm riêng: chỉ đếm cặp **cùng nhóm** đi qua cạnh. Cạnh `(v, cha)` được đi qua bởi cặp `(x, y)` cùng nhóm `g` ⇔ một nút trong cây con của `v`, nút kia ngoài. Số cặp như vậy: `in_g(v) · (tot_g − in_g(v))`.

Ví dụ 3 (sao tâm 0, nhóm `[1,1,4,4]`, gốc 0):

| cạnh | cây con | in₁, in₄ | đóng góp nhóm 1 | đóng góp nhóm 4 |
|---|---|---|---|---|
| 0–1 | {1} | 1, 0 | 1·(2−1) = 1 | 0 |
| 0–2 | {2} | 0, 1 | 0 | 1·(2−1) = 1 |
| 0–3 | {3} | 0, 1 | 0 | 1 |

Tổng `3` ✔ (cặp `(2,3)` đi qua hai cạnh, mỗi cạnh tính 1 lần).

## 5. Cách 1 — Brute Force: BFS từ mọi nút

Từ mỗi nút `u` BFS tính khoảng cách tới mọi nút, cộng các nút cùng nhóm; chia đôi (mỗi cặp đếm hai lần). `O(n²)`.

In [5]:
def brute(n, edges, group):
    _, _, adj = bfs_order(n, edges)
    total = 0
    for u in range(n):
        dist = [-1] * n; dist[u] = 0; dq = deque([u])
        while dq:
            x = dq.popleft()
            for y in adj[x]:
                if dist[y] < 0:
                    dist[y] = dist[x] + 1; dq.append(y)
        total += sum(dist[v] for v in range(n) if group[v] == group[u])
    return total // 2

for n, e, g, exp in examples:
    print(brute(n, e, g), "kỳ vọng", exp)

4 kỳ vọng 4
2 kỳ vọng 2
3 kỳ vọng 3
0 kỳ vọng 0


**Nút thắt:** `O(n²) = 10^{10}`. Đổi sang đếm theo cạnh — mỗi cạnh `O(số nhóm)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Công thức:** `answer = Σ_{v ≠ gốc} Σ_g in_g(v) · (tot_g − in_g(v))`.

**Tính `in_g(v)`:** DP cây con — `in(v) = e_{group[v]} + Σ_{con c} in(c)` (vector độ dài `G`). Duyệt BFS ngược để con trước cha.

**Độ phức tạp:** `O(n · G)` với `G ≤ 20` ⇒ `2·10^6`.

**Hướng khác (theo nhóm):** với mỗi nhóm `g` riêng, chạy một lượt DP đếm `in_g` vô hướng ⇒ `G` lượt `O(n)` — cùng độ phức tạp, ít bộ nhớ hơn.

## 7. Các cách tiếp theo

- **Cách 2:** từng nhóm một — `G` lượt DP cây con, mỗi lượt `O(n)`.
- **Cách 3:** một lượt với **vector đếm** `in[v][g]` (lời giải nộp chính).
- **Cách 4:** **đổi gốc**: với mỗi nhóm, tính `S_g(u)` = tổng khoảng cách từ `u` tới các nút nhóm `g`; đáp án `½ Σ_g Σ_{u ∈ g} S_g(u)`.

### 7.1. Cách 2 — Từng nhóm một

In [6]:
def per_group(n, edges, group):
    order, parent, _ = bfs_order(n, edges)
    total = 0
    for g in set(group):
        tot = sum(1 for x in group if x == g)
        if tot < 2: continue
        cnt = [1 if group[i] == g else 0 for i in range(n)]
        for u in reversed(order):
            p = parent[u]
            if p != -1:
                total += cnt[u] * (tot - cnt[u]); cnt[p] += cnt[u]
    return total

for n, e, g, exp in examples:
    print(per_group(n, e, g), "kỳ vọng", exp)

4 kỳ vọng 4
2 kỳ vọng 2
3 kỳ vọng 3
0 kỳ vọng 0


**Độ phức tạp:** `O(G · n)`, bộ nhớ `O(n)`. Bỏ qua nhóm có < 2 nút.

### 7.2. Cách 3 — Một lượt với vector đếm (lời giải chính)

In [7]:
def vector_pass(n, edges, group):
    order, parent, _ = bfs_order(n, edges)
    ids = {g: i for i, g in enumerate(sorted(set(group)))}
    G = len(ids)
    gid = [ids[g] for g in group]
    tot = [0] * G
    for x in gid: tot[x] += 1
    cnt = [[0] * G for _ in range(n)]
    for i in range(n): cnt[i][gid[i]] = 1
    total = 0
    for u in reversed(order):
        p = parent[u]
        if p == -1: continue
        cu, cp = cnt[u], cnt[p]
        for k in range(G):
            c = cu[k]
            if c:
                total += c * (tot[k] - c)
                cp[k] += c
    return total

for n, e, g, exp in examples:
    print(vector_pass(n, e, g), "kỳ vọng", exp)

4 kỳ vọng 4
2 kỳ vọng 2
3 kỳ vọng 3
0 kỳ vọng 0


**Độ phức tạp Cách 3:** `O(n · G)` thời gian và bộ nhớ.

### 7.3. Cách 4 — Đổi gốc (rerooting)

Với nhóm `g`: `down[v]` = tổng khoảng cách từ `v` tới nút nhóm `g` **trong cây con** của `v`; `S(gốc) = down[gốc]`. Khi chuyển gốc từ cha `p` sang con `v`: `in_g(v)` nút gần hơn 1, `tot_g − in_g(v)` nút xa hơn 1 ⇒ `S(v) = S(p) − in_g(v) + (tot_g − in_g(v))`.

In [8]:
def rerooting(n, edges, group):
    order, parent, _ = bfs_order(n, edges)
    total2 = 0
    for g in set(group):
        tot = sum(1 for x in group if x == g)
        if tot < 2: continue
        cnt = [1 if group[i] == g else 0 for i in range(n)]
        down = [0] * n
        for u in reversed(order):
            p = parent[u]
            if p != -1:
                cnt[p] += cnt[u]; down[p] += down[u] + cnt[u]
        S = [0] * n; S[order[0]] = down[order[0]]
        for u in order[1:]:
            S[u] = S[parent[u]] - cnt[u] + (tot - cnt[u])
        total2 += sum(S[u] for u in range(n) if group[u] == g)
    return total2 // 2

for n, e, g, exp in examples:
    print(rerooting(n, e, g), "kỳ vọng", exp)

4 kỳ vọng 4
2 kỳ vọng 2
3 kỳ vọng 3
0 kỳ vọng 0


In [9]:
class SolutionReroot:
    def interactionCosts(self, n, edges, group):
        return rerooting(n, edges, group)

print([SolutionReroot().interactionCosts(*ex[:3]) for ex in examples])

[4, 2, 3, 0]


**Độ phức tạp Cách 4:** `O(G · n)`. Lợi ích phụ: có luôn `S_g(u)` cho từng nút — trả lời được "tổng chi phí tương tác của nút `u` với nhóm của nó".

## 8. TRACE từng bước

Trace Cách 3 trên cây `0-1, 1-2, 1-3, 0-4` với `group = [1, 2, 1, 2, 1]`.

In [10]:
def trace(n, edges, group):
    order, parent, _ = bfs_order(n, edges)
    ids = {g: i for i, g in enumerate(sorted(set(group)))}; G = len(ids)
    tot = Counter(group); cnt = [Counter({group[i]: 1}) for i in range(n)]
    total = 0
    print("thứ tự BFS:", order, "| tổng mỗi nhóm:", dict(tot))
    for u in reversed(order):
        p = parent[u]
        if p == -1: continue
        add = sum(c * (tot[g] - c) for g, c in cnt[u].items())
        total += add
        print(f"cạnh {u}-{p}: cây con {u} đếm {dict(cnt[u])} -> đóng góp {add}, tổng={total}")
        cnt[p].update(cnt[u])
    return total

e = [[0, 1], [1, 2], [1, 3], [0, 4]]; g = [1, 2, 1, 2, 1]
print("đáp án:", trace(5, e, g), "| vét cạn:", brute(5, e, g))

thứ tự BFS: [0, 1, 4, 2, 3] | tổng mỗi nhóm: {1: 3, 2: 2}
cạnh 3-1: cây con 3 đếm {2: 1} -> đóng góp 1, tổng=1
cạnh 2-1: cây con 2 đếm {1: 1} -> đóng góp 2, tổng=3
cạnh 4-0: cây con 4 đếm {1: 1} -> đóng góp 2, tổng=5
cạnh 1-0: cây con 1 đếm {2: 2, 1: 1} -> đóng góp 2, tổng=7
đáp án: 7 | vét cạn: 7


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def interactionCosts(self, n: int, edges: List[List[int]], group: List[int]) -> int:
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        parent = [-1] * n
        order = [0]
        seen = [False] * n
        seen[0] = True
        for u in order:
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    order.append(v)
        total = 0
        for g in set(group):
            tot = group.count(g)
            if tot < 2:
                continue
            cnt = [1 if x == g else 0 for x in group]
            for u in reversed(order):
                p = parent[u]
                if p != -1:
                    c = cnt[u]
                    if c:
                        total += c * (tot - c)
                        cnt[p] += c
        return total

sol = Solution()
for n, e, g, exp in examples:
    got = sol.interactionCosts(n, e, g)
    print(got, "✔" if got == exp else "✘")

4 ✔
2 ✔
3 ✔
0 ✔


## 10. Vì sao thuật toán đúng

1. **Khoảng cách = số cạnh trên đường đi** ⇒ `Σ_{cặp} dist = Σ_{cạnh} #{cặp có đường đi qua cạnh}` (đổi thứ tự lấy tổng).
2. **Cạnh trên đường đi:** trong cây, cạnh `(v, cha)` nằm trên đường `x ↔ y` ⇔ đúng một trong hai nút thuộc cây con của `v`.
3. **Cùng nhóm:** số cặp cùng nhóm `g` bị cạnh tách = `in_g(v) · (tot_g − in_g(v))`; cộng mọi nhóm.
4. **DP cây con:** duyệt BFS ngược ⇒ khi xử lý `u`, mọi con của `u` đã cộng vào `cnt[u]` ⇒ `cnt[u] = in_g(u)`.
5. **Đổi gốc (Cách 4):** khi dời gốc qua cạnh `(p, v)`, khoảng cách tới các nút trong cây con `v` giảm 1, tới các nút còn lại tăng 1.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS từ mọi nút | `O(n²)` | `O(n)` |
| 2. Từng nhóm | `O(G · n)` | `O(n)` |
| 3. Vector đếm | `O(G · n)` | `O(G · n)` |
| 4. Đổi gốc | `O(G · n)` | `O(n)` |

Đáp án có thể tới `~n³/6 ≈ 1,7·10^{14}` (đường thẳng, một nhóm) — Python không lo tràn.

## 12. Bộ test

In [12]:
def run_all(n, e, g, with_brute=True):
    res = [per_group(n, e, g), vector_pass(n, e, g), rerooting(n, e, g), Solution().interactionCosts(n, e, g)]
    if with_brute:
        res.append(brute(n, e, g))
    return res

for n, e, g, exp in examples:
    assert all(v == exp for v in run_all(n, e, g))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [13]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    (1, [], [5], 0),                                 # một nút
    (4, [[0, 1], [1, 2], [2, 3]], [1, 2, 3, 4], 0),  # mọi nhóm khác nhau
    (4, [[0, 1], [1, 2], [2, 3]], [7, 7, 7, 7], 10), # đường 4 nút: tổng mọi khoảng cách
    (5, [[0, 1], [0, 2], [0, 3], [0, 4]], [1] * 5, 16),
]
for n, e, g, exp in cases:
    r = run_all(n, e, g)
    assert all(v == exp for v in r), (n, e, g, r)
print("Ca biên: OK")

Ca biên: OK


In [14]:
# 12.3 phản ví dụ: tính tổng khoảng cách MỌI cặp rồi nhân tỉ lệ cặp cùng nhóm (sai)
def wrong_proportional(n, edges, group):
    allsum = all_pairs_distance_sum(n, edges)
    same_pairs = sum(c * (c - 1) // 2 for c in Counter(group).values())
    return allsum * same_pairs // (n * (n - 1) // 2) if n > 1 else 0

e, g = [[0, 1], [1, 2], [2, 3]], [1, 2, 2, 1]
print("sai:", wrong_proportional(4, e, g), "| đúng:", Solution().interactionCosts(4, e, g))

sai: 3 | đúng: 4


In [15]:
# 12.4 đối chiếu ngẫu nhiên
def random_tree(rng, n):
    perm = list(range(n)); rng.shuffle(perm)
    return [[perm[rng.randrange(i)], perm[i]] for i in range(1, n)]

rng = random.Random(3786)
for _ in range(300):
    n = rng.randint(1, 30); e = random_tree(rng, n); g = [rng.randint(1, 4) for _ in range(n)]
    r = run_all(n, e, g)
    assert len(set(r)) == 1, (n, e, g, r)
print("300 test ngẫu nhiên: OK")

300 test ngẫu nhiên: OK


In [16]:
# 12.5 test lớn: n = 10^5, 20 nhóm; và đường thẳng một nhóm (đáp án ~ n^3/6)
rng = random.Random(9)
n = 10**5
e = random_tree(rng, n); g = [rng.randint(1, 20) for _ in range(n)]
t0 = time.time(); a = Solution().interactionCosts(n, e, g); t1 = time.time()
print("ngẫu nhiên:", a, "| khớp Cách 3:", a == vector_pass(n, e, g), "| thời gian:", round(t1 - t0, 2), "s")
line = [[i, i + 1] for i in range(n - 1)]
print("đường thẳng 1 nhóm:", Solution().interactionCosts(n, line, [1] * n), "= (n^3 - n)/6 =", (n ** 3 - n) // 6)

ngẫu nhiên: 5054007320 | khớp Cách 3: True | thời gian: 0.39 s
đường thẳng 1 nhóm: 166666666650000 = (n^3 - n)/6 = 166666666650000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm cặp có thứ tự (quên chia 2) ở cách BFS từ mọi nút

BFS từ `u` cộng khoảng cách tới mọi `v` cùng nhóm ⇒ mỗi cặp được cộng hai lần.

In [17]:
def wrong_double(n, edges, group):
    return brute(n, edges, group) * 2               # mô phỏng quên // 2

print("sai:", wrong_double(*examples[0][:3]), "| đúng:", Solution().interactionCosts(*examples[0][:3]))

sai: 8 | đúng: 4


### Lỗi 2 — Dùng `size(v) · (n − size(v))` (mọi cặp) thay vì đếm theo nhóm

In [18]:
print("không phân nhóm:", all_pairs_distance_sum(3, [[0, 1], [1, 2]]), "| đúng với nhóm [3,2,3]:", Solution().interactionCosts(3, [[0, 1], [1, 2]], [3, 2, 3]))

không phân nhóm: 4 | đúng với nhóm [3,2,3]: 2


### Lỗi 3 — DFS đệ quy trên cây sâu `10^5`

Như các bài cây khác: dùng BFS + thứ tự ngược.

In [19]:
import sys
print("giới hạn đệ quy:", sys.getrecursionlimit(), "< 10^5 nút có thể có trên đường thẳng")

giới hạn đệ quy: 1000 < 10^5 nút có thể có trên đường thẳng


### Lỗi 4 — Cộng dồn `cnt` của con vào cha **trước** khi tính đóng góp của cạnh con

Thứ tự đúng: đóng góp của cạnh `(u, cha)` dùng `cnt[u]` hoàn chỉnh (đã gồm mọi con của `u`), sau đó mới cộng vào cha. Duyệt BFS ngược bảo đảm điều này. Nếu duyệt BFS **xuôi** (cha trước con) thì `cnt[u]` chưa đủ.

In [20]:
def wrong_forward(n, edges, group):
    order, parent, _ = bfs_order(n, edges); total = 0
    for g in set(group):
        tot = group.count(g); cnt = [1 if x == g else 0 for x in group]
        for u in order:                                    # SAI: phải duyệt ngược
            p = parent[u]
            if p != -1: total += cnt[u] * (tot - cnt[u]); cnt[p] += cnt[u]
    return total

e = [[0, 1], [1, 2], [2, 3]]
print("sai:", wrong_forward(4, e, [1, 1, 1, 1]), "| đúng:", Solution().interactionCosts(4, e, [1, 1, 1, 1]))

sai: 9 | đúng: 10


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tổng khoảng cách mọi cặp trong cây"** ⇒ đếm đóng góp từng cạnh `s · (n − s)` (LC 834, LC 2242).
- **Có điều kiện trên cặp (cùng nhóm/màu)** với **ít nhóm** ⇒ tách theo nhóm, mỗi nhóm một công thức cạnh.
- **Cần giá trị cho từng nút** ⇒ đổi gốc (LC 834 *Sum of Distances in Tree*).
- Nhãn nhỏ (`≤ 20`) ⇒ độ phức tạp `O(G · n)` chấp nhận được.

## 15. Biến thể của bài

1. **Nhiều nhóm (tới `n`)**: `O(G · n)` quá lớn ⇒ dùng **small-to-large merging** (gộp `Counter` nhỏ vào lớn) hoặc cây ảo (virtual tree) cho từng nhóm.
2. **Cặp khác nhóm**: tổng mọi cặp trừ tổng cùng nhóm.
3. **Trọng số cạnh**: nhân đóng góp với trọng số.
4. **Tổng cho từng nút** (`S_g(u)` với nhóm của `u`): đổi gốc — Cách 4.

In [21]:
# Biến thể 2: tổng khoảng cách các cặp KHÁC nhóm
def different_groups(n, edges, group):
    return all_pairs_distance_sum(n, edges) - Solution().interactionCosts(n, edges, group)

e = [[0, 1], [1, 2], [2, 3]]; g = [1, 2, 1, 2]
bf = sum(abs(i - j) for i in range(4) for j in range(i + 1, 4) if g[i] != g[j])
print(different_groups(4, e, g), "| vét cạn (đường thẳng):", bf)

6 | vét cạn (đường thẳng): 6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mọi nút | Khoảng cách trực tiếp | `O(n²)` | `O(n)` | Kiểm thử |
| 2. Từng nhóm | `G` lượt đếm cây con | `O(Gn)` | `O(n)` | **Lời giải chuẩn**, ít bộ nhớ |
| 3. Vector đếm | Một lượt, mảng `G` | `O(Gn)` | `O(Gn)` | Khi muốn một lượt duyệt |
| 4. Đổi gốc | `S_g(u)` cho mọi nút | `O(Gn)` | `O(n)` | Cần thông tin theo nút |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** Đường `0-1-2-3`, `group = [1,2,1,2]` → ?

**Bài 2 (tính tay).** Sao tâm 0 với 3 lá, mọi nút cùng nhóm.

**Bài 3 (điền TODO).** Viết `sum_of_distances(n, edges)` (LC 834): với mỗi nút, tổng khoảng cách tới mọi nút khác.

**Bài 4 (tự viết lại).** Viết lại lời giải chính với cạnh có trọng số `w`.

**Bài 5 (mini challenge).** Nhóm tới `n` nhãn khác nhau: dùng **small-to-large** (gộp `dict` nhỏ vào lớn) để đạt `O(n log n)`.

In [22]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def sum_of_distances(n, edges):
    # TODO: size, down; S[gốc] = down[gốc]; S[v] = S[p] - size[v] + (n - size[v])
    pass

In [23]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def small_to_large(n, edges, group):
    # TODO: mỗi nút giữ dict nhóm -> số nút trong cây con; đóng góp cạnh = Σ c*(tot - c) trên dict của con
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Nhóm 1: nút 0, 2 (khoảng cách 2). (2) Nhóm 2: nút 1, 3 (khoảng cách 2). (3) Đáp án `4`.

**Bài 2.** (1) 3 cặp (tâm, lá) khoảng cách 1. (2) 3 cặp (lá, lá) khoảng cách 2. (3) `3 + 6 = 9`.

**Bài 3.** (1) Đổi gốc như Cách 4 với một nhóm duy nhất. (2) `S[0] = Σ độ sâu`. (3) LC 834 ví dụ cho `[8,12,6,10,10,10]`.

**Bài 4.** (1) Đóng góp cạnh nhân `w`. (2) Cần lưu trọng số cạnh tới cha. (3) Đối chiếu BFS có trọng số (Dijkstra/DFS).

**Bài 5.** (1) `dict` của con lớn nhất được tái sử dụng. (2) Đóng góp cạnh cần `Σ c (tot − c)` trên dict — duy trì tổng này khi gộp. (3) Mỗi phần tử bị chuyển `O(log n)` lần.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().interactionCosts(4, [[0, 1], [1, 2], [2, 3]], [1, 2, 1, 2]) == 4
assert Solution().interactionCosts(4, [[0, 1], [0, 2], [0, 3]], [1, 1, 1, 1]) == 9
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
# Bài 3 (LC 834)
def sum_of_distances_sol(n, edges):
    order, parent, _ = bfs_order(n, edges)
    size = [1] * n; down = [0] * n
    for u in reversed(order):
        p = parent[u]
        if p != -1: size[p] += size[u]; down[p] += down[u] + size[u]
    S = [0] * n; S[0] = down[0]
    for u in order[1:]:
        S[u] = S[parent[u]] - size[u] + (n - size[u])
    return S

assert sum_of_distances_sol(6, [[0, 1], [0, 2], [2, 3], [2, 4], [2, 5]]) == [8, 12, 6, 10, 10, 10]
print("Bài 3 OK")

Bài 3 OK


In [26]:
# Bài 4: cạnh có trọng số
def weighted(n, edges, weights, group):
    adj = [[] for _ in range(n)]
    for (a, b), w in zip(edges, weights): adj[a].append((b, w)); adj[b].append((a, w))
    parent = [-1] * n; pw = [0] * n; order = [0]; seen = [False] * n; seen[0] = True
    for u in order:
        for v, w in adj[u]:
            if not seen[v]: seen[v] = True; parent[v] = u; pw[v] = w; order.append(v)
    total = 0
    for g in set(group):
        tot = group.count(g); cnt = [1 if x == g else 0 for x in group]
        for u in reversed(order):
            p = parent[u]
            if p != -1: total += pw[u] * cnt[u] * (tot - cnt[u]); cnt[p] += cnt[u]
    return total

def brute_weighted(n, edges, weights, group):
    adj = [[] for _ in range(n)]
    for (a, b), w in zip(edges, weights): adj[a].append((b, w)); adj[b].append((a, w))
    total = 0
    for u in range(n):
        dist = [-1] * n; dist[u] = 0; st = [u]
        while st:
            x = st.pop()
            for y, w in adj[x]:
                if dist[y] < 0: dist[y] = dist[x] + w; st.append(y)
        total += sum(dist[v] for v in range(n) if group[v] == group[u])
    return total // 2

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 15); e = random_tree(rng, n); w = [rng.randint(1, 9) for _ in e]; g = [rng.randint(1, 3) for _ in range(n)]
    assert weighted(n, e, w, g) == brute_weighted(n, e, w, g)
print("Bài 4 OK")

Bài 4 OK


In [27]:
# Bài 5: small-to-large cho số nhóm lớn
def small_to_large_sol(n, edges, group):
    order, parent, _ = bfs_order(n, edges)
    tot = Counter(group)
    bag = [None] * n; acc = [0] * n          # acc[u] = Σ_g c_g (tot_g - c_g) trên bag[u]
    for u in range(n):
        g = group[u]; bag[u] = {g: 1}; acc[u] = tot[g] - 1
    total = 0
    for u in reversed(order):
        p = parent[u]
        if p == -1: continue
        total += acc[u]
        a, b = bag[p], bag[u]; s = acc[p]
        if len(a) < len(b):
            a, b = b, a; s = acc[u]
        for g, c in b.items():
            old = a.get(g, 0); new = old + c
            s += new * (tot[g] - new) - old * (tot[g] - old)
            a[g] = new
        bag[p], acc[p] = a, s
        bag[u] = None
    return total

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 25); e = random_tree(rng, n); g = [rng.randint(1, n) for _ in range(n)]
    assert small_to_large_sol(n, e, g) == brute(n, e, g)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
BFS từ 0 -> order, parent
for g in các nhóm (≥ 2 nút):
    cnt = [x == g for x in group]
    for u in reversed(order):
        if u != gốc:
            total += cnt[u] * (tot_g - cnt[u])   # đóng góp cạnh (u, cha)
            cnt[parent[u]] += cnt[u]
return total
```

**Checklist:**

- [ ] Đếm theo **nhóm**, không phải `size · (n − size)`?
- [ ] Duyệt **ngược** BFS (con trước cha)?
- [ ] Không đệ quy sâu?
- [ ] Cặp không thứ tự (mỗi cặp một lần)?
- [ ] Bỏ qua nhóm có < 2 nút để tiết kiệm?